In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
df= pd.read_csv("./data/london_merged.csv")

In [3]:
df.head()

,timestamp,cnt,t1,t2,hum,wind_speed,weather_code,is_holiday,is_weekend,season
0,2015-01-04 00:00:00,182,3.0,2.0,93.0,6.0,3.0,0.0,1.0,3.0
1,2015-01-04 01:00:00,138,3.0,2.5,93.0,5.0,1.0,0.0,1.0,3.0
2,2015-01-04 02:00:00,134,2.5,2.5,96.5,0.0,1.0,0.0,1.0,3.0
3,2015-01-04 03:00:00,72,2.0,2.0,100.0,0.0,1.0,0.0,1.0,3.0
4,2015-01-04 04:00:00,47,2.0,0.0,93.0,6.5,1.0,0.0,1.0,3.0


In [4]:
df.tail()

,timestamp,cnt,t1,t2,hum,wind_speed,weather_code,is_holiday,is_weekend,season
17409,2017-01-03 19:00:00,1042,5.0,1.0,81.0,19.0,3.0,0.0,0.0,3.0
17410,2017-01-03 20:00:00,541,5.0,1.0,81.0,21.0,4.0,0.0,0.0,3.0
17411,2017-01-03 21:00:00,337,5.5,1.5,78.5,24.0,4.0,0.0,0.0,3.0
17412,2017-01-03 22:00:00,224,5.5,1.5,76.0,23.0,4.0,0.0,0.0,3.0
17413,2017-01-03 23:00:00,139,5.0,1.0,76.0,22.0,2.0,0.0,0.0,3.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17414 entries, 0 to 17413
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   timestamp     17414 non-null  object 
 1   cnt           17414 non-null  int64  
 2   t1            17414 non-null  float64
 3   t2            17414 non-null  float64
 4   hum           17414 non-null  float64
 5   wind_speed    17414 non-null  float64
 6   weather_code  17414 non-null  float64
 7   is_holiday    17414 non-null  float64
 8   is_weekend    17414 non-null  float64
 9   season        17414 non-null  float64
dtypes: float64(8), int64(1), object(1)
memory usage: 1.3+ MB


In [6]:
df.describe().T.style.background_gradient(cmap='GnBu',axis=0)

,count,mean,std,min,25%,50%,75%,max
cnt,17414.000000,1143.101642,1085.108068,0.000000,257.000000,844.000000,1671.750000,7860.000000
t1,17414.000000,12.468091,5.571818,-1.500000,8.000000,12.500000,16.000000,34.000000
t2,17414.000000,11.520836,6.615145,-6.000000,6.000000,12.500000,16.000000,34.000000
hum,17414.000000,72.324954,14.313186,20.500000,63.000000,74.500000,83.000000,100.000000
wind_speed,17414.000000,15.913063,7.894570,0.000000,10.000000,15.000000,20.500000,56.500000
weather_code,17414.000000,2.722752,2.341163,1.000000,1.000000,2.000000,3.000000,26.000000
is_holiday,17414.000000,0.022051,0.146854,0.000000,0.000000,0.000000,0.000000,1.000000
is_weekend,17414.000000,0.285403,0.451619,0.000000,0.000000,0.000000,1.000000,1.000000
season,17414.000000,1.492075,1.118911,0.000000,0.000000,1.000000,2.000000,3.000000


- In time series analysis, the temporal index dictates your entire strategy. Standard generic EDA steps (like random histograms) are less effective here. Instead, we want to uncover how patterns change across different scales of time.

- Before starting our data analysis we must first insure that timestamps is a datetime columns and act as the index for our dataframe this is done for two main reason : 

1- Memory Efficiency & Search Speed: Converting timestamp from an object (string) to datetime64[ns] converts raw text into highly optimized integer nanoseconds under the hood. This makes sorting, filtering, and slicing incredibly fast.

2- Datetime Indexing Power: Once set as the index, students can slice the dataset using intuitive string queries instead of complex logical masks:

In [7]:
# 1. Parse the timestamp column to datetime format
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2. Set 'timestamp' as the DataFrame index
df.set_index('timestamp', inplace=True)

# 3. Sort the index to guarantee strict chronological order
df.sort_index(inplace=True)

# Let's inspect the index type and frequency
print(df.index)

DatetimeIndex(['2015-01-04 00:00:00', '2015-01-04 01:00:00',
               '2015-01-04 02:00:00', '2015-01-04 03:00:00',
               '2015-01-04 04:00:00', '2015-01-04 05:00:00',
               '2015-01-04 06:00:00', '2015-01-04 07:00:00',
               '2015-01-04 08:00:00', '2015-01-04 09:00:00',
               ...
               '2017-01-03 14:00:00', '2017-01-03 15:00:00',
               '2017-01-03 16:00:00', '2017-01-03 17:00:00',
               '2017-01-03 18:00:00', '2017-01-03 19:00:00',
               '2017-01-03 20:00:00', '2017-01-03 21:00:00',
               '2017-01-03 22:00:00', '2017-01-03 23:00:00'],
              dtype='datetime64[ns]', name='timestamp', length=17414, freq=None)


- First let's explore the Target Variable across Multiple Time Scales. In time series, we always explore the target variable (cnt) first to understand its rhythmic behaviors (daily, weekly, and seasonal cycles) before we look at how weather, holidays, or wind speeds affect it.

In [8]:
# 3. Cast categorical/boolean float columns to integers
categorical_cols = ['weather_code', 'is_holiday', 'is_weekend', 'season']
df[categorical_cols] = df[categorical_cols].astype(int)

In [9]:

# ==========================================
# STEP 1: CREATE TIME COLUMNS FROM THE INDEX
# ==========================================
# Extract hour (0 to 23)
df['hour'] = df.index.hour

# Extract day name (Monday, Tuesday, etc.)
df['day_name'] = df.index.day_name()


# ==========================================
# STEP 2: CALCULATE THE AVERAGES
# ==========================================
# 1. Macro (Monthly average)
monthly_avg = df['cnt'].resample('ME').mean()

# 2. Medium (Day of week average - ordered correctly from Monday to Sunday)
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly_avg = df.groupby('day_name')['cnt'].mean().reindex(day_order)

# 3. Micro (Hourly average for weekdays vs weekends)
weekday_hourly = df[df['is_weekend'] == 0].groupby('hour')['cnt'].mean()
weekend_hourly = df[df['is_weekend'] == 1].groupby('hour')['cnt'].mean()


# ==========================================
# STEP 3: BUILD THE PLOTS
# ==========================================
# Create a grid with 1 row and 3 columns
fig = make_subplots(
    rows=1, 
    cols=3, 
    subplot_titles=('1. Monthly Trend', '2. Daily Trend', '3. Hourly Trend')
)

# Plot 1: Monthly (Macro Scale)
fig.add_trace(
    go.Scatter(x=monthly_avg.index, y=monthly_avg.values, name='Monthly Mean'), 
    row=1, col=1
)

# Plot 2: Day of the Week (Medium Scale)
fig.add_trace(
    go.Bar(x=weekly_avg.index, y=weekly_avg.values, name='Daily Mean'), 
    row=1, col=2
)

# Plot 3: Hourly (Micro Scale - comparing Weekday vs Weekend)
fig.add_trace(
    go.Scatter(x=weekday_hourly.index, y=weekday_hourly.values, name='Weekdays'), 
    row=1, col=3
)
fig.add_trace(
    go.Scatter(x=weekend_hourly.index, y=weekend_hourly.values, name='Weekends'), 
    row=1, col=3
)

# Show the dashboard with a clean white background
fig.update_layout(title='Bike Share Exploratory Analysis', template='plotly_white')
fig.show()

- Key Findings: Interpreting the Target Variable (`cnt`)

By exploring our target variable across three distinct time scales, we can clearly see how human behavior and weather shape bike-sharing demand in London:

- 1. Macro Scale: Monthly Trend (Yearly Seasonality)
* **Observation:** Demand follows a repeating annual wave, peaking dramatically in July/August (~1,600 avg rentals/hr) and dropping to its lowest point in January (~800 avg rentals/hr).
* **Takeaway:** Demand is heavily temperature-dependent. Warm summer months drive outdoor activity, while cold, wet winters drastically suppress ridership. 
* **Trend:** The summer peak of 2016 is slightly higher than 2015, indicating a gradual, positive year-over-year growth in overall bike adoption.

- 2. Medium Scale: Daily Trend (Weekly Seasonality)
* **Observation:** Average hourly rentals are highest during mid-week workdays (Tuesday through Thursday) and drop visibly on Saturday and Sunday.
* **Takeaway:** This indicates that the **primary user base consists of daily commuters**. If the service were mainly used for leisure, we would expect weekends to dominate.

- 3. Micro Scale: Hourly Trend (Daily Seasonality)
This plot unmasks the differing behavioral signatures of weekdays versus weekends:
* **Weekdays (Mon–Fri):** Exhibit a sharp **bimodal (double-peak) distribution** peaking at **8:00 AM** and **5:00 PM – 6:00 PM**. This represents classic morning and evening rush-hour office commutes.
* **Weekends (Sat–Sun):** Exhibit a smooth, **unimodal (single-peak) bell curve** peaking between **12:00 PM and 3:00 PM**. This represents relaxed, midday recreational and leisure riding.



- 💡 Why This Matters for Modeling
A simple average will fail here. To build an accurate forecasting model, our algorithms must know:
1. **The Hour of the Day** (Micro)
2. **The Day of the Week** (Medium)
3. **The Month/Season** (Macro)

These temporal features must be explicitly engineered to help our machine learning models capture these overlapping rhythms!

--- 

#####  Classical Seasonal Decomposition ($y_t \rightarrow T_t + S_t + I_t$)

Now that we have visually explored our target variable across different time scales, we can perform a formal **Classical Seasonal Decomposition**. This mathematical process allows us to isolate and study the individual underlying forces driving our time series.

- 1. Mathematical Formulation

A time series $y_t$ can be modeled as a combination of three main components:
1. **Trend ($T_t$):** The long-term movement or baseline direction of the data over time.
2. **Seasonal ($S_t$):** Regular, periodic patterns that repeat over a known, fixed window (e.g., daily, weekly, or annually).
3. **Residual / Irregular ($I_t$):** The random, unpredictable noise or "leftovers" after removing the trend and seasonal patterns.

Depending on how these components interact, we model them using one of two functional forms:

- A. Additive Model (Used here)
We use this model when the amplitude of the seasonal fluctuations stays relatively constant, regardless of the level of the trend.
$$y_t = T_t + S_t + I_t$$

- B. Multiplicative Model
We use this model when the seasonal swings increase or decrease proportionally with the overall trend of the data.
$$y_t = T_t \times S_t \times I_t$$



- Why We Resample to Daily Averages First

Before running the decomposition, we resample our hourly data to **daily averages** using `.resample('D').mean()`. 

* **Why?** Our raw dataset contains multiple nested seasonalities (hourly, weekly, and yearly). Classical algorithms like `seasonal_decompose` can only model **one** seasonal period at a time. 
* By aggregating to daily averages, we filter out the high-frequency hourly "noise" and focus purely on the macro **yearly temperature cycle** (setting our frequency period to $365$ days).



- Student Guide: How to Read the 4 Subplots

1. **Observed (Raw Daily):** The real-world historical timeline of daily bike rentals. It is highly volatile and difficult for statistical models to interpret directly.
2. **Trend (Orange):** The stripped, smooth baseline. Notice how it acts as a moving filter, showing whether the overall bike-share program is gaining or losing long-term popularity.
3. **Seasonal (Green):** A mathematically perfect, repeating annual cycle. This is the pure reflection of the weather: rising in June/July and bottoming out in December/January.
4. **Residuals (Red):** The unexplainable noise. If you spot a massive downward spike here, it represents an external anomaly (e.g., severe storm, public holiday, or a transit strike) that the regular calendar or trend could not predict.

In [13]:

from statsmodels.tsa.seasonal import seasonal_decompose

# 1. Resample hourly data to daily averages
daily_series = df['cnt'].resample('D').mean()

# 2. FILL THE CALENDAR GAPS (The Fix!)
# This fills any NaN values created by days with missing records
daily_series = daily_series.interpolate(method='linear')



In [12]:
# Change period from 365 to 7 to model weekly cycles
decomposition = seasonal_decompose(daily_series, model='additive', period=7)

trend = decomposition.trend
seasonal = decomposition.seasonal
residual = decomposition.resid

# Re-plot
fig = make_subplots(
    rows=4, 
    cols=1, 
    shared_xaxes=True,
    subplot_titles=('Observed Data', 'Trend (7-Day Smooth)', 'Seasonal (Weekly Pattern)', 'Residuals (Noise)')
)

fig.add_trace(go.Scatter(x=daily_series.index, y=daily_series.values, name='Observed'), row=1, col=1)
fig.add_trace(go.Scatter(x=trend.index, y=trend.values, name='Trend'), row=2, col=1)
fig.add_trace(go.Scatter(x=seasonal.index, y=seasonal.values, name='Seasonal'), row=3, col=1)
fig.add_trace(go.Scatter(x=residual.index, y=residual.values, name='Residuals'), row=4, col=1)

fig.update_layout(height=800, title='Time Series Decomposition (Weekly Cycle)', template='plotly_white')
fig.show()

- Interpreting our 7-Day Weekly Decomposition

Now that we have adjusted our seasonal period to $7$ (representing a weekly cycle), our decomposition is mathematically stable and highly informative:

1. **Observed Data:** Our raw daily averages, showing clear high-frequency weekly spikes riding on top of the long-term annual wave.
2. **Trend (7-Day Smooth):** By using a short 7-day window, we preserve almost our entire timeline (losing only 3 days of data at each boundary). This line smoothly tracks the transitions between summer peaks and winter troughs without any weekday noise.
3. **Seasonal (Weekly Pattern):** A clean, repeating weekly profile. Because the timeline spans 2 years, this plot shows over 100 identical weekly cycles packed together.
4. **Residuals (Noise):** This represents the "leftovers" (e.g., unexpected weather anomalies, public holidays, or transit disruptions). Notice how the residuals are mostly packed around 0, with a few major outliers—such as the large positive spike in July 2015.

In [15]:
from statsmodels.tsa.stattools import adfuller

# Run the Augmented Dickey-Fuller test on our daily averages
result = adfuller(daily_series)

# Print the essential results simply
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")
print(f"Critical Value (5%): {result[4]['5%']:.4f}")

ADF Statistic: -1.3886
p-value: 0.5877
Critical Value (5%): -2.8656


- Statistical Stationarity (Augmented Dickey-Fuller Test)

To build reliable forecasting models (like ARIMA), our time series must be **stationary**—meaning its statistical properties (mean, variance, and autocorrelation) do not change over time.

- Our ADF Test Results
* **ADF Statistic:** `-1.3886`
* **p-value:** `0.5877`
* **Critical Value (5%):** `-2.8656`

- How to Interpret This:
1. **The Hypothesis:**
   * **Null Hypothesis ($H_0$):** The series has a unit root (it is **non-stationary**).
   * **Alternative Hypothesis ($H_1$):** The series has no unit root (it is **stationary**).
2. **The Verdict:**
   * Since our $p$-value ($0.5877$) is **greater than $0.05$**, we **fail to reject the null hypothesis**.
   * Our ADF statistic ($-1.3886$) is also much less negative than the $5\%$ critical value ($-2.8656$).
   * **Conclusion:** Our series is **non-stationary**. It has a prominent long-term seasonal trend that causes the mean to drift over time.

- The Solution: First-Order Differencing ($\Delta y_t$)
To remove this drift and stabilize the mean, we will compute the differences between consecutive days:
$$\Delta y_t = y_t - y_{t-1}$$

In [16]:
# 1. Compute the first-order difference (y_t - y_{t-1})
# This removes the trend and stabilizes the mean
differenced_series = daily_series.diff().dropna()

# 2. Re-run the Augmented Dickey-Fuller test on our differenced data
diff_result = adfuller(differenced_series)

# 3. Print the new results
print("--- After First-Order Differencing ---")
print(f"New ADF Statistic: {diff_result[0]:.4f}")
print(f"New p-value: {diff_result[1]:.4f}")
print(f"New Critical Value (5%): {diff_result[4]['5%']:.4f}")

--- After First-Order Differencing ---
New ADF Statistic: -8.3120
New p-value: 0.0000
New Critical Value (5%): -2.8656


- 🎉 Success: We Achieved Stationarity!

By calculating the difference between consecutive days ($\Delta y_t = y_t - y_{t-1}$), we have completely stabilized our time series:

* **New ADF Statistic:** `-8.3120` (Extremely negative!)
* **New p-value:** `0.0000` (Way below our 0.05 threshold)
* **Critical Value (5%):** `-2.8656`

- 💡 Interpretation:
Because our $p$-value is $0.0000$, we **strongly reject the Null Hypothesis ($H_0$)** of non-stationarity. 
* The strong downward trend has been successfully removed.
* The mean and variance of our series are now constant over time.
* **Our series is now stationary!** This means we have officially cleared the entry barrier to start building classical forecasting models.

In [18]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf, pacf

# 1. Calculate ACF and PACF values (40 lags)
lags = 40
acf_values = acf(differenced_series, nlags=lags)
pacf_values = pacf(differenced_series, nlags=lags)

# 2. Create side-by-side subplots
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=('Autocorrelation (ACF)', 'Partial Autocorrelation (PACF)')
)

# 3. Add Bar charts to represent the lag spikes (no complex loops needed!)
fig.add_trace(go.Bar(x=list(range(lags+1)), y=acf_values, name='ACF'), row=1, col=1)
fig.add_trace(go.Bar(x=list(range(lags+1)), y=pacf_values, name='PACF'), row=1, col=2)

# 4. Show the interactive plot
fig.update_layout(height=400, template='plotly_white', showlegend=False)
fig.show()

- How to Decode ACF and PACF Plots

Before building a forecasting model, we use these two plots to analyze how the current day's data ($y_t$) relates to past days ($y_{t-k}$):

1. **ACF (Autocorrelation):** Measures the total correlation (direct + indirect) between today and $k$ days ago. 
   * *What we see:* Strong positive spikes at lags 7, 14, 21, and 28. This proves a **weekly cyclical pattern** still exists!
2. **PACF (Partial Autocorrelation):** Measures only the *direct* correlation between today and $k$ days ago, removing any middle-day influences.
   * *What we see:* Significant negative spikes for the first 5-6 lags, followed by a sudden drop.
   
- Our Model Choice Strategy:
* Because the **PACF cuts off** and the **ACF decays gradually**, our non-seasonal data behaves like an **Autoregressive (AR)** process.
* Because of the recurring spikes at every multiple of 7, we will use a **SARIMA** model to capture both daily momentum and weekly habits.

--- 
- Module 3: Classical Time Series Modeling (SARIMA)

We have verified that our daily bike rental series is **non-stationary** due to a drifting mean, and we successfully stabilized it using **first-order differencing**. 

To model and forecast this stationary data, we rely on two classical linear modeling frameworks: **ARIMA** and **SARIMA**. Let's understand how they work and how to configure them using our diagnostics.



- The Baseline: ARIMA(p, d, q)

**ARIMA** stands for **A**uto**R**egressive **I**ntegrated **M**oving **A**verage. It is designed to capture short-term trends and momentum in time series data that **do not** contain repeating seasonal cycles.

It consists of three components:

* **AR ($p$) — Autoregressive:** Uses the relationship between an observation and a number of lagged (past) observations ($y_{t-1}, y_{t-2}, \dots$). It captures the "momentum" of the series.
* **I ($d$) — Integrated:** Represents the number of differencing steps required to make the time series stationary (to remove trend and drift). Since we differenced our series once, $d = 1$.
* **MA ($q$) — Moving Average:** Models the relationship between an observation and residual errors from a moving average model applied to lagged observations ($e_{t-1}, e_{t-2}, \dots$). It captures sudden shocks or random anomalies.

$$\hat{y}_t = c + \phi_1 y_{t-1} + \dots + \phi_p y_{t-p} + \theta_1 e_{t-1} + \dots + \theta_q e_{t-q}$$



- The Upgrade: SARIMA(p, d, q)(P, D, Q)s

While ARIMA is highly effective, it has a major limitation: **it cannot capture repeating seasonal cycles**. If we apply a standard ARIMA to our bike data, the model will fail to realize that weekends consistently behave differently than weekdays.

To solve this, **SARIMA** adds a seasonal layer directly to the ARIMA math:

$$\text{SARIMA} \equiv \underbrace{(p, d, q)}_{\text{Non-Seasonal Part}} \times \underbrace{(P, D, Q)_s}_{\text{Seasonal Part}}$$

### How the Seasonal Parameters Work:
* **$s$ (Seasonal Period):** The number of time steps in a single seasonal cycle. Because our data represents daily averages with a weekly cycle, **$s = 7$**.
* **$P$ (Seasonal Autoregressive):** Looks at the relationship between today and the same day in previous cycles (e.g., today's rental numbers vs. last week's same day, $y_{t-7}$).
* **$D$ (Seasonal Integration):** The number of seasonal differences needed to remove seasonal trends (e.g., subtracting $y_{t-7}$ from $y_t$).
* **$Q$ (Seasonal Moving Average):** Models the relationship between today's value and past seasonal errors (e.g., $e_{t-7}$).


- Our Model Choice: Why SARIMA is the Winner

Based on our previous diagnostics, we must select **SARIMA** for three crucial reasons:

1.  **Massive Weekly Seasonality:** Our ACF plot exhibited dominant, statistically significant spikes at lags **7, 14, 21, and 28**. Standard ARIMA would treat these predictable weekly cycles as random noise, reducing forecasting accuracy.
2.  **Explicit Period ($s = 7$):** Because our data aggregates daily averages, our seasonal cycle length is exactly 7 days.
3.  **Interaction of Trends:** Bike rentals depend heavily on the day of the week (commuters on weekdays vs. leisure riders on weekends) riding on top of a larger, long-term seasonal wave. SARIMA is uniquely engineered to handle both components simultaneously.

In [19]:
import statsmodels.api as sm

# Define model parameters based on our diagnostics:
# Non-seasonal: (p=1, d=1, q=1) -> Daily adjustments
# Seasonal: (P=1, D=1, Q=1, s=7) -> Weekly adjustments
model = sm.tsa.statespace.SARIMAX(
    daily_series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

# Fit the model to the data
sarima_results = model.fit(disp=False)

# Display the training summary and statistical diagnostics
print(sarima_results.summary())

                                     SARIMAX Results                                     
Dep. Variable:                               cnt   No. Observations:                  731
Model:             SARIMAX(1, 1, 1)x(1, 1, 1, 7)   Log Likelihood               -4866.908
Date:                           Tue, 14 Jul 2026   AIC                           9743.816
Time:                                   21:31:41   BIC                           9766.670
Sample:                               01-04-2015   HQIC                          9752.642
                                    - 01-03-2017                                         
Covariance Type:                             opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.2952      0.035      8.428      0.000       0.227       0.364
ma.L1         -0.9104      0.018    -50.205

###  Step 8: Interpreting the SARIMA Model Results

After fitting our $SARIMA(1, 1, 1)(1, 1, 1)_7$ model, we must analyze the statistical output to verify if the parameters are meaningful.

#### 1. Parameter Analysis (The Middle Table)
* **`ar.L1` (Non-seasonal AR, $p=0.000$):** Highly significant. A positive coefficient ($0.2952$) indicates today's demand maintains a positive momentum from yesterday.
* **`ma.L1` (Non-seasonal MA, $p=0.000$):** Highly significant. The model is actively correcting for short-term shocks.
* **`ma.S.L7` (Seasonal MA, $p=0.000$):** Highly significant. This verifies that adjusting for weekly cyclical shocks is crucial for accuracy.
* **`ar.S.L7` (Seasonal AR, $p=0.362$):** **Not statistically significant!** Because the $p$-value is greater than $0.05$, this specific parameter is not contributing useful information to the model. In a real-world iteration, we would drop this to simplify our model to $SARIMA(1, 1, 1)(0, 1, 1)_7$.

#### 2. Diagnostic Testing (The Bottom Section)
* **Ljung-Box Test (`Prob(Q) = 0.76`):** **SUCCESS.** Because the $p$-value is $> 0.05$, there is no significant leftover autocorrelation in our residuals. They behave like random white noise!
* **Heteroskedasticity (`Prob(H) = 0.39`):** **SUCCESS.** The error variance remains stable over the duration of the dataset.
* **Jarque-Bera Test (`Prob(JB) = 0.00`):** **INFO.** The residuals are not perfectly normally distributed, showing some heavy tails (outliers). This is expected for real-world physical demand (impacted by anomalous events like holidays or storms).

# 📈 Module 4: Diagnostics and Forecasting

Now that our SARIMA model is trained, we need to:
1. **Validate our residuals** to ensure the model isn't leaving any valuable pattern behind in the errors.
2. **Generate future forecasts** and plot them alongside our historical data.

---

## 🛠️ Step 1: Residual Diagnostics

A healthy model produces residuals that look like **white noise**: they should be centered around zero, have constant variance, and exhibit no remaining autocorrelation. Let's plot our residuals to visually confirm this!

In [20]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Retrieve the model's residuals (ignoring the very first few values affected by differencing)
residuals = sarima_results.resid[10:]

# Create a diagnostic dashboard
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Residuals Over Time</b>', '<b>Distribution of Residuals</b>')
)

# 1. Residuals over time (should look like random fluctuations around 0)
fig.add_trace(
    go.Scatter(x=residuals.index, y=residuals, mode='lines', line=dict(color='#8884d8'), name='Residuals'),
    row=1, col=1
)
fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=1)

# 2. Histogram of residuals (should approximate a normal bell curve)
fig.add_trace(
    go.Histogram(x=residuals, marker_color='#82ca9d', name='Density'),
    row=1, col=2
)

fig.update_layout(height=400, template='plotly_white', showlegend=False, title_text="<b>SARIMA Residual Diagnostics</b>")
fig.show()

---

## 🔮 Step 2: Out-of-Sample Forecasting

Now for the grand finale: let's use our fitted model to forecast bike rentals **30 days into the future**! 

To make this look ultra-professional for our dashboard, we will plot:
1. The **historical observed data** (past 60 days to keep the visual clean).
2. The **predicted forecast line** for the next 30 days.
3. A **95% Confidence Interval band** (the shaded area showing the model's margin of uncertainty).

In [21]:
import pandas as pd

# 1. Generate forecasts for the next 30 days
forecast_steps = 30
forecast_obj = sarima_results.get_forecast(steps=forecast_steps)

# Extract forecast mean and confidence intervals
forecast_mean = forecast_obj.predicted_mean
confidence_intervals = forecast_obj.conf_int()

# 2. Slice the last 60 days of historical data for clean visualization
historical_subset = daily_series[-60:]

# 3. Create the interactive Plotly visualization
fig = go.Figure()

# Plot historical data
fig.add_trace(go.Scatter(
    x=historical_subset.index, 
    y=historical_subset, 
    mode='lines', 
    name='Historical Observed', 
    line=dict(color='#1f77b4', width=2)
))

# Plot forecast mean
fig.add_trace(go.Scatter(
    x=forecast_mean.index, 
    y=forecast_mean, 
    mode='lines+markers', 
    name='30-Day Forecast', 
    line=dict(color='#ff7f0e', width=2, dash='dash')
))

# Plot Upper Confidence Limit (UCL)
fig.add_trace(go.Scatter(
    x=confidence_intervals.index, 
    y=confidence_intervals.iloc[:, 1], 
    mode='lines', 
    line=dict(width=0),
    showlegend=False,
    name='Upper Bound'
))

# Plot Lower Confidence Limit (LCL) filled to the Upper Bound
fig.add_trace(go.Scatter(
    x=confidence_intervals.index, 
    y=confidence_intervals.iloc[:, 0], 
    mode='lines', 
    fill='tonexty', # Fills the area between upper and lower bounds
    fillcolor='rgba(255, 127, 14, 0.2)', 
    line=dict(width=0),
    name='95% Confidence Interval'
))

# Layout Styling
fig.update_layout(
    title='<b>🚲 30-Day Future Bike Rental Forecast</b>',
    xaxis_title='Date',
    yaxis_title='Daily Average Rentals',
    template='plotly_white',
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

# 🎯 Module 5: Evaluating Diagnostics & the Future Forecast

Now that we have successfully visualized our results, let's perform a comprehensive evaluation of our model's performance.

---

## 1. Residual Diagnostics: Are We Left with White Noise?

A forecasting model is only complete when its errors (residuals) contain **no remaining patterns**. If patterns remain, it means we left predictive information on the table.

* **Fluctuation Around Zero (Left Plot):** Our residuals bounce rapidly and randomly around the $0$ line. This proves that our non-seasonal and seasonal differencing steps ($d=1, D=1$) completely removed the long-term trends and weekly cyclical structures.
* **The Normal Distribution (Right Plot):** The histogram of our residuals displays a highly symmetric, bell-shaped curve centered at $0$. This visual alignment corresponds to our passing **Ljung-Box test ($p = 0.76$)**, confirming our errors behave like independent white noise.
* **Why Did We Fail Jarque-Bera?** You will notice a few isolated residual spikes (e.g., July 2015 and January 2017). These represent extreme anomalous days (unusual weather, sudden public events, or holidays) that violate perfect normality. In real-world data science, this is entirely expected and acceptable.

---

## 2. Reading the 30-Day Forecast: The SARIMA Impact

Looking at our future predictions, we can clearly see the mathematical power of our model:

* **Preserving the Weekly Rhythm:** Rather than predicting a flat average line, the orange forecast path tracks a distinct weekly wave. This demonstrates that our seasonal parameters are actively mimicking weekly passenger habits.
* **Adapting to Winter Cycles:** The model successfully predicts a significant overall drop in daily rentals during January compared to the warmer months. It has learned the broader yearly temperature relationships.
* **The Confidence Interval (The Peach Shaded Band):** Notice how the shaded boundaries widen as we forecast further into January and February. This represents the cumulative growth of statistical uncertainty—predicting tomorrow is simple; predicting four weeks from now carries a wider margin of error.